In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout, SimpleRNN, LSTM, LayerNormalization, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import re
import scipy.io
import psutil
import os
import math
import cmath 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler

#infile_path = 'post_total_data.csv'
infile_path = 'post_dist_bb_1GHZ_csv.csv'
df = pd.read_csv(infile_path)

xin_real = df['xI_real']
xin_imag = df['xI_imag']
yout_real = df['yout_real']
yout_imag = df['yout_imag']

#t_ns = np.arange(1,300001)
t_ns = np.arange(1,30001)

y_arr = np.column_stack((xin_real, xin_imag))
# youtn = np.array(youtn)
# youtn_real = youtn.real
# youtn_imag = youtn.imag
X_arr = np.column_stack((yout_real, yout_imag))

X = pd.DataFrame(X_arr, columns=['yout_real', 'yout_imag'])
y = pd.DataFrame(y_arr, columns=['xin_real', 'xin_imag'])



#Old sequences
N_train = 24_000

X_new = X.iloc[N_train:]
#y_new = y.iloc[240000:300001]
print("Shape of X_new:", X_new.shape)
# Keep the first 90,000 rows in both dataframes:
X = X.iloc[:N_train]
y = y.iloc[:N_train]


print("Shape of y:", y.shape)


Shape of X_new: (6000, 2)
Shape of y: (24000, 2)


In [3]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Build the model
model = Sequential([
    Input(shape=(2,)),  # Explicitly defining the input shape
    Dense(32, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(2)  # Output layer with 2 neurons for xI_real and xI_imag
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, validation_split=0.2, verbose=1)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)


y_pred = model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f'Test Loss: {test_loss}')
print(f'Test accuracy: {test_accuracy}')
print(f'Test RMSE: {rmse}')

print("Training complete")

Epoch 1/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9368 - loss: 0.0166 - val_accuracy: 0.9596 - val_loss: 0.0037
Epoch 2/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9590 - loss: 0.0036 - val_accuracy: 0.9589 - val_loss: 0.0035
Epoch 3/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9600 - loss: 0.0036 - val_accuracy: 0.9596 - val_loss: 0.0035
Epoch 4/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9575 - loss: 0.0034 - val_accuracy: 0.9615 - val_loss: 0.0034
Epoch 5/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9563 - loss: 0.0035 - val_accuracy: 0.9589 - val_loss: 0.0035
Epoch 6/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9621 - loss: 0.0034 - val_accuracy: 0.9620 - val_loss: 0.0036
Epoch 7/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9607 - loss: 0.0034 - val_accuracy: 0.9596 - val_loss: 0.0037
Epoch 8/10
480/480 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9571 - loss: 0.0034 - val_accuracy: 0.

In [5]:
# Function to parse complex numbers, including those with scientific notation from excel csv file
def parse_complex_number(s):
    s = s.replace("i", "j")  
    # Updated regex to handle scientific notation
    match = re.match(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([-+]\d*\.?\d+(?:[eE][-+]?\d+)?)j", s.replace(" ", ""))
    if match:
        real_part = float(match.group(1))
        imag_part = float(match.group(2))
        return real_part, imag_part
    else:
        raise ValueError(f"Malformed complex number: {s}")


# Predict using the model
#LSTM new seq here
#y_pred_s = model.predict(X_new_seq)
y_new_pred = model.predict(X_new) 
#y_new_pred = y_scaler.inverse_transform(y_pred_s)
#y_new_pred = model.predict(y_new_seq) 

# Create the DataFrame with predictions
df_predict = pd.DataFrame({
    'xI_predict_real': y_new_pred[:, 0],
    'xI_predict_imag': y_new_pred[:, 1]
})


# Create complex numbers
df_predict['xI_predict'] = (df_predict['xI_predict_real'] + 1j * df_predict['xI_predict_imag']).astype(np.complex128)

# Prepare data for MATLAB file generation
data_to_save = {'xI_predict': df_predict['xI_predict'].values}


output_mat_file_path = 'C:\\Users\\Luke McCubbin\\Box\\NN PA DPD\\Postdistortion_study_UCSD_10_5_2025\\ANN_TD0_V1GHz.mat'
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f'Predictions saved to {output_mat_file_path}')

188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 970us/step
Predictions saved to C:\Users\Luke McCubbin\Box\NN PA DPD\Postdistortion_study_UCSD_10_5_2025\ANN_TD0_V1GHz.mat
